# 04 - Retrieval-depth evaluation v2

## Purpose

This notebook records the adaptive version 2 retrieval-depth evaluation. The
version 1 adequacy gate showed that candidate depths ending at 10 were too
narrow, so the protocol was amended before any expanded search or additional
judgement was performed.

Version 2 has three deliberately separate actions:

1. reuse the exact version 1 query matrix, search locally to rank 30 and migrate
   the 400 verified rank 1-20 judgements;
2. judge only the 200 new rank 21-30 candidate pairs through the same blinded
   interface;
3. score all 600 judgements offline using the unchanged selection rule and the
   new adequacy gate.

The adaptive decision, frozen lineage and isolated outputs support the
dissertation requirements for methodological transparency, reproducibility,
critical justification and responsible evaluation-data handling.


## Evaluation and leakage boundary

- This is retrieval evaluation, not answer generation.
- Version 2 follows the protocol amendment frozen in commit `98f3e92`.
- The 20 questions, query embeddings, corpus chunks and FAISS index are
  unchanged from version 1.
- The query matrix is reused byte for byte, so version 2 makes no embedding API
  request.
- The version 2 pool contains ranks 1-30. Candidate depths end at 20 and ranks
  21-30 remain a diagnostic tail.
- The 400 rank 1-20 grades and notes were migrated only after every frozen
  candidate field matched.
- The annotation view displays only a question and one candidate chunk. It
  hides rank, score, identifiers, candidate-depth membership and provenance.
- Ground-truth answers and model-generated relevance recommendations remain
  unavailable during annotation.
- The same 20 benchmark questions are used for calibration and later model
  evaluation, so the selected depth remains benchmark-tuned rather than an
  independent holdout estimate.

The preparation, annotation and scoring cells are disabled by default.
Therefore, running all cells from a fresh kernel does not create or alter an
evaluation artifact.


In [ ]:
from __future__ import annotations

from html import escape
from pathlib import Path
from tempfile import TemporaryDirectory
import json
import os
import sys

from IPython.display import HTML, clear_output, display
import numpy as np


def locate_project_root() -> Path:
    """Locate the repository from its root or notebooks directory."""

    candidates = [Path.cwd().resolve(), Path.cwd().resolve().parent]
    for candidate in candidates:
        if (
            (candidate / "pyproject.toml").is_file()
            and (candidate / "configs/retrieval-evaluation-config-v2.json").is_file()
        ):
            return candidate
    raise RuntimeError(
        "Run this notebook from the repository root or its notebooks directory."
    )


def resolve_project_path(root: Path, relative_path: str) -> Path:
    """Resolve a configured path without allowing it to escape the project."""

    path = (root / relative_path).resolve()
    if not path.is_relative_to(root):
        raise RuntimeError(f"Configured path leaves the project: {relative_path}")
    return path


def env_file_defines_key(path: Path, key: str) -> bool:
    """Check whether a key name exists without reading or printing its value."""

    if not path.is_file():
        return False
    for raw_line in path.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        name = line.split("=", 1)[0].strip()
        if name.startswith("export "):
            name = name.removeprefix("export ").strip()
        if name == key:
            return True
    return False


PROJECT_ROOT = locate_project_root()
SOURCE_ROOT = (PROJECT_ROOT / "src").resolve()
if not SOURCE_ROOT.is_dir():
    raise RuntimeError(f"Project source directory is missing: {SOURCE_ROOT}")

source_root_string = str(SOURCE_ROOT)
if source_root_string not in sys.path:
    sys.path.insert(0, source_root_string)

from geotech_rag.retrieval_evaluation import (
    calculate_file_sha256,
    load_retrieval_evaluation_config,
    load_retrieval_evaluation_inputs,
    score_retrieval_judgements,
    serialise_json_lines,
)
from geotech_rag.retrieval_evaluation_v2 import (
    prepare_retrieval_evaluation_v2,
)


CONFIG_PATH = PROJECT_ROOT / "configs/retrieval-evaluation-config-v2.json"
LINEAGE_PATH = PROJECT_ROOT / "configs/retrieval-evaluation-v2-lineage.json"
NOTEBOOK_PATH = PROJECT_ROOT / "notebooks/04_retrieval_depth_evaluation_v2.ipynb"

print("Project root:", PROJECT_ROOT)
print("Source root added to Python path:", SOURCE_ROOT)
print("Configuration:", CONFIG_PATH.relative_to(PROJECT_ROOT))
print("Lineage:", LINEAGE_PATH.relative_to(PROJECT_ROOT))
print("Ground truth accessed: False")


## 1. Frozen v2 protocol and input preflight

This preflight loads the committed version 2 configuration and lineage
contract. It validates the unchanged question, corpus, mapping and FAISS inputs
without making an API request or displaying question or chunk text.


In [ ]:
# Load the isolated v2 configuration and its frozen v1-to-v2 lineage.
config = load_retrieval_evaluation_config(CONFIG_PATH)
inputs = load_retrieval_evaluation_inputs(CONFIG_PATH, PROJECT_ROOT)
lineage = json.loads(LINEAGE_PATH.read_text(encoding="utf-8"))
outputs = config["outputs"]

# Resolve every output through the same project-boundary check used in v1.
QUERY_MATRIX_PATH = resolve_project_path(
    PROJECT_ROOT,
    outputs["query_embedding_matrix_relative_path"],
)
CANDIDATE_POOL_PATH = resolve_project_path(
    PROJECT_ROOT,
    outputs["ranked_candidate_pool_relative_path"],
)
JUDGEMENT_PATH = resolve_project_path(
    PROJECT_ROOT,
    outputs["blinded_judgement_file_relative_path"],
)
SUMMARY_PATH = resolve_project_path(
    PROJECT_ROOT,
    outputs["summary_relative_path"],
)
METRICS_PATH = resolve_project_path(
    PROJECT_ROOT,
    outputs["metrics_relative_path"],
)
METRICS_TABLE_PATH = resolve_project_path(
    PROJECT_ROOT,
    outputs["metrics_table_relative_path"],
)

# Derive the expected counts from the frozen contracts rather than inserting
# independent values into the notebook.
question_count = len(inputs.question_records)
pool_depth = config["search"]["judgement_pool_depth"]
expected_candidate_count = question_count * pool_depth
expected_reused_count = lineage["judgement_reuse"][
    "reused_judgement_count"
]
expected_new_count = lineage["judgement_reuse"][
    "new_judgement_count"
]

if (
    lineage["target_v2"]["configuration_id"]
    != config["configuration_id"]
):
    raise RuntimeError(
        "The v2 configuration and lineage identifiers differ."
    )

if (
    lineage["target_v2"]["configuration_sha256"]
    != calculate_file_sha256(CONFIG_PATH)
):
    raise RuntimeError(
        "The v2 configuration fingerprint differs from the lineage."
    )

if expected_reused_count + expected_new_count != expected_candidate_count:
    raise RuntimeError(
        "The v2 reuse counts do not equal the candidate count."
    )

print("Retrieval-evaluation v2 preflight:")
print("  Configuration ID:", config["configuration_id"])
print("  Questions:", question_count)
print("  Corpus chunks:", len(inputs.chunk_records))
print("  FAISS vectors:", inputs.faiss_index.ntotal)
print("  Candidate depths:", config["search"]["candidate_depths"])
print("  Maximum candidate depth:", config["search"]["maximum_candidate_depth"])
print("  Judgement pool depth:", pool_depth)
print("  Expected candidate pairs:", expected_candidate_count)
print("  Expected migrated judgements:", expected_reused_count)
print("  Expected new judgements:", expected_new_count)
print(
    "  Query-embedding API request required:",
    lineage["query_embedding_reuse"]["api_request_required"],
)
print("  Credential value displayed: False")
print("  Question or chunk text displayed: False")
print("  Ground truth accessed: False")
print("Retrieval-evaluation v2 preflight: PASSED")


## 2. Guarded local v2 preparation

Version 2 reuses the exact version 1 query matrix and searches the existing
FAISS index locally to rank 30. It validates the 400 rank 1-20 candidate pairs
before transferring their existing human grades and notes. The 200 rank 21-30
records remain ungraded.

The production preparation was already completed from committed implementation
`0b4a990` and independently audited before this notebook was created. Keep
`RUN_V2_PREPARATION = False`. The preparation function also refuses to replace
existing outputs.


In [ ]:
RUN_V2_PREPARATION = False

if not RUN_V2_PREPARATION:
    print("Retrieval-depth v2 preparation: NOT RUN")
    print("The audited v2 preparation artifacts are reused as they exist.")
else:
    # This branch is available only for a clean first preparation. The
    # implementation refuses to overwrite any existing v2 artifact.
    preparation_result = prepare_retrieval_evaluation_v2(
        CONFIG_PATH,
        LINEAGE_PATH,
        PROJECT_ROOT,
    )
    print("Retrieval-depth v2 preparation: PASSED")
    print("  Candidate pairs:", preparation_result.candidate_count)
    print(
        "  Reused judgements:",
        preparation_result.reused_judgement_count,
    )
    print(
        "  New judgements:",
        preparation_result.new_judgement_count,
    )
    print(
        "  Query matrix SHA-256:",
        preparation_result.query_embedding_matrix_sha256,
    )
    print(
        "  Candidate pool SHA-256:",
        preparation_result.ranked_candidate_pool_sha256,
    )
    print(
        "  Judgement file SHA-256:",
        preparation_result.blinded_judgement_file_sha256,
    )
    print(
        "  Summary SHA-256:",
        preparation_result.summary_sha256,
    )
    print("  API request made: False")
    print("  Ground truth accessed: False")


## 3. V2 preparation artifact audit

This cell checks the v2 shapes, counts, fingerprints and blinded schema. It
reports only safe metadata and does not print question text, chunk text, rank,
score, provenance or the grade distribution.

The separately tracked preparation audit fixes the exact pre-annotation
fingerprints and confirms that all 400 migrated values match version 1.


In [ ]:
preparation_paths = [
    QUERY_MATRIX_PATH,
    CANDIDATE_POOL_PATH,
    JUDGEMENT_PATH,
    SUMMARY_PATH,
]

if not all(path.is_file() for path in preparation_paths):
    print("Preparation artifact audit: NOT RUN")
    print("Run the guarded paid preparation first.")
else:
    summary = json.loads(SUMMARY_PATH.read_text(encoding="utf-8"))
    query_matrix = np.load(QUERY_MATRIX_PATH, allow_pickle=False)
    with CANDIDATE_POOL_PATH.open("r", encoding="utf-8") as handle:
        candidate_records = [json.loads(line) for line in handle if line.strip()]
    with JUDGEMENT_PATH.open("r", encoding="utf-8") as handle:
        judgement_records = [json.loads(line) for line in handle if line.strip()]

    expected_judgement_fields = {
        "chunk_text",
        "configuration_id",
        "judgement_id",
        "judgement_note",
        "judgement_schema_version",
        "query_text",
        "relevance_grade",
    }
    hidden_judgement_fields = {
        "question_id",
        "chunk_id",
        "question_position",
        "index_position",
        "rank",
        "similarity_score",
        "parent_record_id",
        "source_id",
        "pdf_page_index",
        "pdf_page_number",
        "printed_page_number",
    }

    if query_matrix.shape != (
        question_count,
        config["query_embedding"]["dimensions"],
    ):
        raise RuntimeError(f"Unexpected query matrix shape: {query_matrix.shape}")
    if query_matrix.dtype != np.float32 or not query_matrix.flags.c_contiguous:
        raise RuntimeError("Query matrix does not follow the float32 C-contiguous contract.")
    if not np.isfinite(query_matrix).all():
        raise RuntimeError("Query matrix contains a non-finite value.")
    if len(candidate_records) != expected_candidate_count:
        raise RuntimeError("Candidate count differs from the frozen protocol.")
    if len(judgement_records) != expected_candidate_count:
        raise RuntimeError("Judgement count differs from the frozen protocol.")
    if any(set(record) != expected_judgement_fields for record in judgement_records):
        raise RuntimeError("A blinded judgement record has unexpected fields.")
    if any(hidden_judgement_fields.intersection(record) for record in judgement_records):
        raise RuntimeError("A rank or provenance field leaked into the judgement file.")
    if summary["question_embedding_matrix_sha256"] != calculate_file_sha256(
        QUERY_MATRIX_PATH
    ):
        raise RuntimeError("Query matrix fingerprint differs from the summary.")
    if summary["ranked_candidate_pool_sha256"] != calculate_file_sha256(
        CANDIDATE_POOL_PATH
    ):
        raise RuntimeError("Candidate-pool fingerprint differs from the summary.")

    if summary.get("query_embedding_reused") is not True:
        raise RuntimeError("The summary does not confirm matrix reuse.")
    if summary.get("prefix_match_passed") is not True:
        raise RuntimeError("The summary does not confirm prefix equality.")
    if (
        summary.get("completed_reused_judgement_count")
        != expected_reused_count
    ):
        raise RuntimeError("The summary has the wrong migrated count.")
    if summary.get("new_unjudged_count") != expected_new_count:
        raise RuntimeError("The summary has the wrong new-judgement count.")
    if summary.get("api_request_made") is not False:
        raise RuntimeError("The summary reports an API request.")

    query_norms = np.linalg.norm(query_matrix, axis=1)
    completed_count = sum(
        record["relevance_grade"] is not None for record in judgement_records
    )

    print("V2 preparation artifact audit:")
    print("  Query matrix shape:", query_matrix.shape)
    print("  Query matrix dtype:", query_matrix.dtype)
    print("  Query matrix C-contiguous:", query_matrix.flags.c_contiguous)
    print("  Finite values:", bool(np.isfinite(query_matrix).all()))
    print("  Minimum norm:", float(query_norms.min()))
    print("  Maximum norm:", float(query_norms.max()))
    print("  Candidate records:", len(candidate_records))
    print("  Blinded judgement records:", len(judgement_records))
    print("  Completed judgements:", completed_count)
    print("  Migrated at preparation:", expected_reused_count)
    print("  New at preparation:", expected_new_count)
    print("  Hidden rank, score and provenance fields present: False")
    print("  Question or chunk text displayed: False")
    print("  Ground truth accessed: False")
    print("V2 preparation artifact audit: PASSED")


## 4. Blinded relevance judgement helper

Use the same frozen grades as version 1:

- `0` - irrelevant: the chunk does not help answer the question;
- `1` - supporting: the chunk gives related context, a definition or a useful
  intermediate fact, but it does not directly provide the main evidence;
- `2` - direct: the chunk directly contains the relationship, procedure, data,
  equation or explanation needed for the question.

Grades 1 and 2 require a short evidence note. The helper searches for records
whose grade is `None`, so the 400 migrated records are never presented or
modified by the annotation loop. Only the 200 new records are eligible.

The private file is saved atomically after every judgment. Question and chunk
text are cleared when the session ends. Do not save or commit the notebook
while a private annotation prompt is visible.


In [ ]:
EXPECTED_JUDGEMENT_FIELDS = {
    "chunk_text",
    "configuration_id",
    "judgement_id",
    "judgement_note",
    "judgement_schema_version",
    "query_text",
    "relevance_grade",
}


def load_local_judgements() -> list[dict[str, object]]:
    """Load the private blinded file without displaying its text."""

    if not JUDGEMENT_PATH.is_file():
        raise RuntimeError("The blinded judgement file does not exist yet.")
    with JUDGEMENT_PATH.open("r", encoding="utf-8") as handle:
        records = [json.loads(line) for line in handle if line.strip()]
    if len(records) != expected_candidate_count:
        raise RuntimeError("Unexpected number of blinded judgement records.")
    if any(set(record) != EXPECTED_JUDGEMENT_FIELDS for record in records):
        raise RuntimeError("A blinded judgement record has unexpected fields.")
    if len({record["judgement_id"] for record in records}) != len(records):
        raise RuntimeError("Blinded judgement identifiers are not unique.")
    return records


def save_local_judgements(records: list[dict[str, object]]) -> None:
    """Atomically replace the private judgement file after validation."""

    if len(records) != expected_candidate_count:
        raise RuntimeError("Refusing to save an incomplete judgement dataset.")
    if any(set(record) != EXPECTED_JUDGEMENT_FIELDS for record in records):
        raise RuntimeError("Refusing to save a record with unexpected fields.")

    JUDGEMENT_PATH.parent.mkdir(parents=True, exist_ok=True)
    with TemporaryDirectory(dir=JUDGEMENT_PATH.parent) as temporary_directory:
        temporary_path = Path(temporary_directory) / JUDGEMENT_PATH.name
        temporary_path.write_bytes(serialise_json_lines(records))
        with temporary_path.open("r", encoding="utf-8") as handle:
            reloaded = [json.loads(line) for line in handle if line.strip()]
        if reloaded != records:
            raise RuntimeError("Judgements changed during the save round trip.")
        os.replace(temporary_path, JUDGEMENT_PATH)


def judgement_progress() -> tuple[int, int]:
    """Return completed and total counts without exposing private content."""

    records = load_local_judgements()
    completed = sum(record["relevance_grade"] is not None for record in records)
    return completed, len(records)


def annotation_session(max_items: int = 10) -> None:
    """Judge up to max_items private pairs and save after every response."""

    if isinstance(max_items, bool) or not isinstance(max_items, int) or max_items < 1:
        raise ValueError("max_items must be a positive integer.")

    processed = 0
    try:
        while processed < max_items:
            records = load_local_judgements()
            next_position = next(
                (
                    position
                    for position, record in enumerate(records)
                    if record["relevance_grade"] is None
                ),
                None,
            )
            if next_position is None:
                break

            record = records[next_position]
            completed = sum(
                value["relevance_grade"] is not None for value in records
            )
            clear_output(wait=True)
            display(
                HTML(
                    "<h3>Blinded retrieval judgement</h3>"
                    f"<p>Progress before save: {completed} / {len(records)}</p>"
                    "<h4>Question</h4>"
                    f"<pre style='white-space:pre-wrap'>{escape(str(record['query_text']))}</pre>"
                    "<h4>Candidate chunk</h4>"
                    f"<pre style='white-space:pre-wrap'>{escape(str(record['chunk_text']))}</pre>"
                    "<p><strong>0</strong> irrelevant, "
                    "<strong>1</strong> supporting, "
                    "<strong>2</strong> direct</p>"
                )
            )

            raw_grade = input("Grade 0, 1, 2, or q to stop: ").strip().lower()
            if raw_grade == "q":
                break
            if raw_grade not in {"0", "1", "2"}:
                print("Invalid grade. This pair was not changed.")
                input("Press Enter to continue: ")
                continue

            grade = int(raw_grade)
            note: str | None = None
            if grade in {1, 2}:
                note = input("Short evidence note: ").strip()
                if not note:
                    print("Grades 1 and 2 require a note. This pair was not changed.")
                    input("Press Enter to continue: ")
                    continue

            record["relevance_grade"] = grade
            record["judgement_note"] = note
            save_local_judgements(records)
            processed += 1
    finally:
        clear_output(wait=False)
        if JUDGEMENT_PATH.is_file():
            completed, total = judgement_progress()
            print("Annotation session closed safely.")
            print("  Judgements saved in this session:", processed)
            print("  Completed judgements:", completed)
            print("  Remaining judgements:", total - completed)
            print("  Total judgements:", total)
            print("  Private question or chunk text retained in cell output: False")


print("Blinded annotation helper: READY")
print("No judgement was changed by defining these functions.")


## 5. Guarded additional annotation session

Change `RUN_ANNOTATION_SESSION` to `True` only while judging a small block. Ten
pairs per session provides regular checkpoints. Enter `q` to stop early, then
restore the flag to `False` before saving the notebook.

The initial progress is 400 of 600 because those 400 records were migrated from
version 1. Only the remaining 200 records require a new human judgment.

Do not consult ranks, scores, candidate-depth membership, provenance,
ground-truth answers or model-generated recommendations while judging.


In [ ]:
RUN_ANNOTATION_SESSION = False
ANNOTATION_BLOCK_SIZE = 10

if not RUN_ANNOTATION_SESSION:
    print("Blinded annotation session: NOT RUN")
    if JUDGEMENT_PATH.is_file():
        completed, total = judgement_progress()
        print("  Completed judgements:", completed)
        print("  Remaining judgements:", total - completed)
        print("  Total judgements:", total)
else:
    annotation_session(max_items=ANNOTATION_BLOCK_SIZE)


## 6. V2 completion audit

This audit reports only completed, remaining and total counts together with the
current private file fingerprint. It does not display private text or the grade
distribution. Version 2 is complete only when all 600 records have a grade.


In [ ]:
if not JUDGEMENT_PATH.is_file():
    print("Completion audit: NOT RUN")
    print("The blinded judgement file does not exist yet.")
else:
    completed, total = judgement_progress()
    print("Blinded judgement completion audit:")
    print("  Completed judgements:", completed)
    print("  Remaining judgements:", total - completed)
    print("  Total judgements:", total)
    print("  Current judgement SHA-256:", calculate_file_sha256(JUDGEMENT_PATH))
    print("  Question or chunk text displayed: False")
    print("  Ground truth accessed: False")
    if completed == total:
        print("Blinded judgement completion audit: PASSED")
    else:
        print("Blinded judgement completion audit: INCOMPLETE")


## 7. Guarded offline v2 scoring

Scoring is allowed only after all 600 judgments are complete. It validates the
private files, calculates metrics for candidate depths 1, 2, 3, 4, 5, 8, 10,
15 and 20, and applies the unchanged selection rule:

1. maximise the number of questions with direct evidence;
2. if tied, maximise the number with useful evidence;
3. if still tied, select the smallest `k`.

The v2 adequacy gate triggers if a question without direct evidence in ranks
1-20 gains direct evidence in ranks 21-30. If it triggers, no retrieval depth
is selected and another versioned protocol decision is required.

This step is offline, makes no API request and does not load ground-truth
answers. Leave the flag disabled until the completion audit passes.


In [ ]:
RUN_OFFLINE_SCORING = False

if not RUN_OFFLINE_SCORING:
    print("Offline retrieval scoring: NOT RUN")
    print("Set RUN_OFFLINE_SCORING = True only after all judgements are complete.")
else:
    completed, total = judgement_progress()
    if completed != total:
        raise RuntimeError(
            f"Cannot score incomplete judgements: {completed} / {total} complete."
        )
    metrics_result = score_retrieval_judgements(
        CONFIG_PATH,
        PROJECT_ROOT,
        overwrite=False,
    )
    print("Offline retrieval scoring: PASSED")
    print("  Completed judgements:", metrics_result.completed_judgement_count)
    print(
        "  Candidate-range adequacy gate triggered:",
        metrics_result.candidate_range_adequacy_gate_triggered,
    )
    print("  Selected retrieval depth k:", metrics_result.selected_depth_k)
    print("  Metrics:", metrics_result.metrics_path.relative_to(PROJECT_ROOT))
    print("  Metrics SHA-256:", metrics_result.metrics_sha256)
    print("  Metrics table:", metrics_result.metrics_table_path.relative_to(PROJECT_ROOT))
    print("  Metrics table SHA-256:", metrics_result.metrics_table_sha256)
    print("  API request made during scoring: False")
    print("  Ground truth accessed: False")


## 8. Text-free v2 metric review

After scoring, this cell displays the aggregated v2 metric rows, adequacy-gate
result and selection status. The tracked results contain fingerprints and
aggregated numbers only, without question or chunk text.


In [ ]:
if not METRICS_PATH.is_file() or not METRICS_TABLE_PATH.is_file():
    print("Metric review: NOT RUN")
    print("Run guarded offline scoring after completing every judgement.")
else:
    metrics = json.loads(METRICS_PATH.read_text(encoding="utf-8"))
    metrics_text = METRICS_PATH.read_text(encoding="utf-8")

    if "query_text" in metrics_text or "chunk_text" in metrics_text:
        raise RuntimeError("Tracked metrics contain a forbidden text field name.")

    print("Retrieval-depth metric review:")
    print("  Selected retrieval depth k:", metrics["selected_depth_k"])
    print("  Selection status:", metrics["selection_status"])
    print(
        "  Candidate-range adequacy gate triggered:",
        metrics["candidate_range_adequacy_gate"]["triggered"],
    )
    print("  Completed judgements:", metrics["completed_judgement_count"])
    print("  Metrics SHA-256:", calculate_file_sha256(METRICS_PATH))
    print("  Metrics table SHA-256:", calculate_file_sha256(METRICS_TABLE_PATH))
    print("  Question or chunk text displayed: False")
    print("  Ground truth accessed: False")
    print()
    print(METRICS_TABLE_PATH.read_text(encoding="utf-8"))
    print("Retrieval-depth metric review: PASSED")


## 9. Reproducibility and reporting boundary

Version 2 is an adaptive extension created after the version 1 adequacy gate
triggered. It must be reported as such and must not be presented as part of the
original frozen version 1 protocol.

If the v2 adequacy gate does not trigger, the unchanged selection rule chooses
the smallest depth that preserves the maximum direct-evidence coverage and then
the maximum useful-evidence coverage. If the gate triggers, the retrieval depth
remains unresolved.

The dissertation should report the version 1 result, the reason for expansion,
the v2 candidate depths, the rank 21-30 diagnostic tail, the exact migration
controls, the final gate result and the limitation that the same 20 questions
were used for calibration.

Before committing this notebook:

- restore `RUN_V2_PREPARATION`, `RUN_ANNOTATION_SESSION` and
  `RUN_OFFLINE_SCORING` to `False`;
- clear any private annotation output;
- validate the notebook JSON;
- confirm that no question or chunk text remains in saved output;
- keep the private pool, matrix and judgment files ignored by Git.
